### This notebook shows the use cases of the Experiment decorator on a trivial example.

In [2]:
# import the Experiment and ExperimentViewer classes from the empirical package
from empirical import Experiment
from empirical.viewer import ExperimentViewer

# other imports
import numpy as np

import time

As stated in `README.md`, this class can be used when dealing with iterative functions for various numerical purposes. Usually, they have a list of parameters, a return value and a loop. At each iteration, one might want to see in console certain outputs (such error function evaluations, different metrics, etc.). A simple example of such a function is stated below:

In [3]:
def numerical_experiment(num_iterations, decay_rate):
    steps = np.arange(num_iterations)
    noise_1 = np.random.normal(0, 0.2, size=num_iterations)
    noise_2 = np.random.normal(0, 0.5, size=num_iterations)
    
    error_1_vec = np.exp(-decay_rate * steps) + noise_1
    error_2_vec = 1.5 * np.exp(-decay_rate * steps) + noise_2
    
    result = []
    
    for i in range(num_iterations):
        
        # some heavyweight compute
        start_time = time.perf_counter()
        
        time.sleep(np.abs(noise_1[i]))
        result.append(error_1_vec[i] + error_2_vec[i])
        
        end_time = time.perf_counter()
        
        # various prints, e.g.,
        print(f' -> [Iteration {i + 1}] Error1: {error_1_vec[i]:.4f}   |   Error2: {error_2_vec[i]:.4f}        Time: {end_time - start_time}') 
        
    return result

In [4]:
params = {
    "num_iterations": 10,
    "decay_rate": 0.75
}

result = numerical_experiment(**params)

 -> [Iteration 1] Error1: 0.6964   |   Error2: 2.0925        Time: 0.3086273339577019
 -> [Iteration 2] Error1: 0.2565   |   Error2: 0.6000        Time: 0.22101737512275577
 -> [Iteration 3] Error1: -0.2109   |   Error2: -0.4003        Time: 0.4367434591986239
 -> [Iteration 4] Error1: 0.1969   |   Error2: 0.3540        Time: 0.09398091724142432
 -> [Iteration 5] Error1: -0.2801   |   Error2: 0.1896        Time: 0.33489891700446606
 -> [Iteration 6] Error1: 0.0243   |   Error2: 0.4593        Time: 0.000960999634116888
 -> [Iteration 7] Error1: -0.1806   |   Error2: 1.2161        Time: 0.1967638754285872
 -> [Iteration 8] Error1: -0.0368   |   Error2: 0.1277        Time: 0.047092583030462265
 -> [Iteration 9] Error1: 0.0913   |   Error2: -1.5180        Time: 0.09139962494373322
 -> [Iteration 10] Error1: 0.0618   |   Error2: -0.4003        Time: 0.06388458283618093


A first example of how Empirical comes into play is the reduced complexity of the code for intermediate prints:

In [5]:
@Experiment(name = "Numerical_Experiment", save_results = False)
def numerical_experiment(num_iterations, decay_rate):
    steps = np.arange(num_iterations)
    noise_1 = np.random.normal(0, 0.2, size=num_iterations)
    noise_2 = np.random.normal(0, 0.5, size=num_iterations)
    
    error_1_vec = np.exp(-decay_rate * steps) + noise_1
    error_2_vec = 1.5 * np.exp(-decay_rate * steps) + noise_2
    
    result = []
    
    for i in range(num_iterations):
        
        # some heavyweight compute
        time.sleep(np.abs(noise_1[i]))

        result.append(error_1_vec[i] + error_2_vec[i])
        
        # yield the intermediary parameters instead
        yield{
            "error_1": error_1_vec[i],
            "error_2": error_2_vec[i]
        }
        
    return result

In [6]:
params = {
    "num_iterations": 10,
    "decay_rate": 0.75
}

result = numerical_experiment(**params)

Starting experiment 'Numerical_Experiment'.
[Wrapper] Iteration 0 -> Time 0.1557s | error_1: 1.1498 | error_2: 1.7394
[Wrapper] Iteration 1 -> Time 0.0572s | error_1: 0.5266 | error_2: 0.7324
[Wrapper] Iteration 2 -> Time 0.0596s | error_1: 0.1686 | error_2: 0.2363
[Wrapper] Iteration 3 -> Time 0.0758s | error_1: 0.0346 | error_2: -0.6823
[Wrapper] Iteration 4 -> Time 0.4195s | error_1: -0.3647 | error_2: -0.1531
[Wrapper] Iteration 5 -> Time 0.0751s | error_1: -0.0466 | error_2: -0.1577
[Wrapper] Iteration 6 -> Time 0.4137s | error_1: 0.4198 | error_2: -0.0173
[Wrapper] Iteration 7 -> Time 0.4229s | error_1: 0.4231 | error_2: 0.5157
[Wrapper] Iteration 8 -> Time 0.2117s | error_1: 0.2091 | error_2: 0.1040
[Wrapper] Iteration 9 -> Time 0.3489s | error_1: 0.3450 | error_2: 0.2666


Moreover, most of the time, this types of functions might require lot of time and compute. Thus, storage on disk might be convenient. Several problems arise:
- results must be organised efficiently
- the input parameters must be stored as well
- metadata about the experiment must be tracked
- the reproducibility of the code must be assured

All of this while also keeping the code clean, with an emphasis on the function design and mathematical or methodological aspects of the experiment.

In [7]:
@Experiment(name = "Numerical_Experiment", save_results = True)
def numerical_experiment(num_iterations, decay_rate):
    steps = np.arange(num_iterations)
    noise_1 = np.random.normal(0, 0.2, size=num_iterations)
    noise_2 = np.random.normal(0, 0.5, size=num_iterations)
    
    error_1_vec = np.exp(-decay_rate * steps) + noise_1
    error_2_vec = 1.5 * np.exp(-decay_rate * steps) + noise_2
    
    result = []
    
    for i in range(num_iterations):
        
        # some heavyweight compute
        time.sleep(np.abs(noise_1[i]))

        result.append(error_1_vec[i] + error_2_vec[i])
        
        # yield the intermediary parameters instead
        yield{
            "error_1": error_1_vec[i],
            "error_2": error_2_vec[i]
        }
        
    return result

In [8]:
params = {
    "num_iterations": 100,
    "decay_rate": 0.75
}

result = numerical_experiment(**params)

Starting experiment 'Numerical_Experiment'. Saved in 'results/Numerical_Experiment/20260820_214832_cd04743'
[Wrapper] [Iteration 0] -> Time 0.0028s | error_1: 1.0020 | error_2: 1.8218
[Wrapper] [Iteration 1] -> Time 0.1057s | error_1: 0.3718 | error_2: 0.7625
[Wrapper] [Iteration 2] -> Time 0.0718s | error_1: 0.2916 | error_2: 0.1082
[Wrapper] [Iteration 3] -> Time 0.2813s | error_1: 0.3825 | error_2: 0.4234
[Wrapper] [Iteration 4] -> Time 0.0435s | error_1: 0.0109 | error_2: -0.6914
[Wrapper] [Iteration 5] -> Time 0.1526s | error_1: 0.1711 | error_2: -0.8374
[Wrapper] [Iteration 6] -> Time 0.0857s | error_1: -0.0695 | error_2: 0.0438
[Wrapper] [Iteration 7] -> Time 0.1823s | error_1: -0.1720 | error_2: -0.6260
[Wrapper] [Iteration 8] -> Time 0.0929s | error_1: -0.0877 | error_2: 0.2332
[Wrapper] [Iteration 9] -> Time 0.0541s | error_1: -0.0525 | error_2: 0.8275
[Wrapper] [Iteration 10] -> Time 0.0425s | error_1: 0.0385 | error_2: 0.2036
[Wrapper] [Iteration 11] -> Time 0.0709s | error

We see below that a `results` folder was created, containing a subfolder of the given Experiment name a which in turn contains another subfolder for the experiment ID along with its generated files.

In [10]:
!ls
!ls results 
!ls results/Numerical_Experiment
!ls results/Numerical_Experiment/20260820_214832_cd04743

experiment_example.ipynb results                  runner_example.ipynb
Numerical_Experiment
20260820_214832_cd04743
history.json              output.log                result.txt
metrics.json              params.json               uncommitted_changes.patch


These results can be easily inspected using the ExperimentViewer class:

In [12]:
experiment = ExperimentViewer("results/Numerical_Experiment/20260820_214832_cd04743")

In [13]:
# print experiment metadata
experiment.summary()



 ---------------------------- EXPERIMENT: Numerical_Experiment / 20260820_214832_cd04743 -----------------------------------------
 -> Start date: 2026-08-20 21:48:32 - End date: 2026-08-20 21:48:47  |  Status: SUCCESS
 -> Time: 00:00:14 (14.7332s)
 -> Peak RAM Usage: 0.09 MB
 -> Git Commit Hash: cd04743 (Uncommitted changes: True)

Parameters:
 -> num_iterations = 100
 -> decay_rate = 0.75


In [14]:
# the returned result
experiment.result

'[np.float64(2.8238796476661783), np.float64(1.1342912247539954), np.float64(0.399783635156504), np.float64(0.8058886973661334), np.float64(-0.6805366143733313), np.float64(-0.6663609444133427), np.float64(-0.02569778424761999), np.float64(-0.7980391658924438), np.float64(0.14550992979775834), np.float64(0.7749717757055646), np.float64(0.2421370648867644), np.float64(-0.20213565992904747), np.float64(0.33196710383104033), np.float64(0.07895664628706985), np.float64(-0.14978031791930924), np.float64(0.3976227264540316), np.float64(-0.17670120891110794), np.float64(0.39397564498539017), np.float64(-0.18303970407298853), np.float64(0.9969960046228266), np.float64(0.3398197693576011), np.float64(0.4630427632789673), np.float64(0.2943159541358021), np.float64(0.4539711249059125), np.float64(-1.4482181517936394), np.float64(0.16938724445827893), np.float64(-0.30904808437187137), np.float64(0.4569813091590756), np.float64(0.5552006191923441), np.float64(-0.3664147147545356), np.float64(-0.448

In [15]:
# the intermediary yielded values, as well as iteration duration
experiment.history

,error_1,error_2,step_duration_s
0,1.002039,1.821840,0.002807
1,0.371783,0.762509,0.105664
2,0.291626,0.108157,0.071816
3,0.382484,0.423404,0.281258
4,0.010908,-0.691445,0.043526
...,...,...,...
95,-0.218721,-0.776496,0.221892
96,0.198285,0.194479,0.203477
97,0.004326,0.177779,0.005531
98,0.009851,0.271830,0.013222


In [16]:
# show the last N lines of the log
experiment.show_log(20)

[Wrapper] [Iteration 80] -> Time 0.0199s | error_1: 0.0186 | error_2: -0.0860
[Wrapper] [Iteration 81] -> Time 0.2491s | error_1: -0.2440 | error_2: 0.4281
[Wrapper] [Iteration 82] -> Time 0.0509s | error_1: 0.0458 | error_2: -0.7798
[Wrapper] [Iteration 83] -> Time 0.0568s | error_1: -0.0517 | error_2: -0.4302
[Wrapper] [Iteration 84] -> Time 0.0143s | error_1: 0.0114 | error_2: -0.2929
[Wrapper] [Iteration 85] -> Time 0.5993s | error_1: -0.5942 | error_2: -0.0417
[Wrapper] [Iteration 86] -> Time 0.2392s | error_1: 0.2341 | error_2: -0.7393
[Wrapper] [Iteration 87] -> Time 0.0704s | error_1: -0.0653 | error_2: -0.4082
[Wrapper] [Iteration 88] -> Time 0.0542s | error_1: 0.0491 | error_2: 0.2589
[Wrapper] [Iteration 89] -> Time 0.0164s | error_1: 0.0131 | error_2: 0.1864
[Wrapper] [Iteration 90] -> Time 0.2088s | error_1: 0.2037 | error_2: -0.1618
[Wrapper] [Iteration 91] -> Time 0.0720s | error_1: -0.0670 | error_2: -0.1349
[Wrapper] [Iteration 92] -> Time 0.3285s | error_1: -0.3233 | 

Moreover, if applicable, the Experiment class saves the latest commit hash as well as diff of the latest code. It can be reinstantiated in a new folder using the `restore_code_state` function from the ExperimentViewer class:

In [17]:
experiment.restore_code_state()

Cloning repository to: /Users/tudorpistol/Empirical/examples/results/Numerical_Experiment/20260820_214832_cd04743/restored_code...
Checking out commit: cd04743...
Applying patch: uncommitted_changes.patch...
Commit and patch applied successfully.

 Code restored in:
/Users/tudorpistol/Empirical/examples/results/Numerical_Experiment/20260820_214832_cd04743/restored_code


In [18]:
!ls results/Numerical_Experiment/20260820_214832_cd04743/restored_code

empirical        examples         requirements.txt setup.py


For more edge cases and error handling, refer to the source code.